# MCP Demo

Unleashing the power of MCP.

This Demo is intended to showcase the capabilities of MCP integrating different systems that can orchestrate a full user experience to booking a Flight.

In [1]:
!pip install -r requirements.txt

In [2]:
import logging

def init_logger() -> logging.Logger:
    logging.basicConfig(
        level=logging.INFO,  # logging.DEBUG,
        format="\x1b[90m[%(levelname)s]\x1b[0m %(message)s"
    )
    return logging.getLogger()

In [4]:
from langchain_mcp_tools import (
    convert_mcp_to_langchain_tools,
    McpServersConfig,
)

mcp_servers: McpServersConfig = {
    "filesystem": {
        "command": "npx",
        "args": [
            "-y",
            "@modelcontextprotocol/server-filesystem",
            "."  # path to a directory to allow access to
        ],
        # "cwd": "/tmp"  # the working dir to be use by the server
    },
    "fetch": {
        "command": "/Users/miguel.romero/.local/bin/uvx",
        "args": [
            "mcp-server-fetch"
        ]
    },
    "weather": {
        "command": "npx",
        "args": [
            "-y",
            "@h1deya/mcp-server-weather"
        ]
    },
    # "weather": {
    #     "url": f"http://localhost:{sse_server_port}/sse"
    # },
    # "weather": {
    #     "url": f"ws://localhost:{ws_server_port}/message"
    # },
}

tools, cleanup = await convert_mcp_to_langchain_tools(
    mcp_servers,
    init_logger(),
)

[INFO] MCP server "filesystem": initializing with: {'command': 'npx', 'args': ['-y', '@modelcontextprotocol/server-filesystem', '.']}
[INFO] MCP server "fetch": initializing with: {'command': '/Users/miguel.romero/.local/bin/uvx', 'args': ['mcp-server-fetch']}
[INFO] MCP server "weather": initializing with: {'command': 'npx', 'args': ['-y', '@h1deya/mcp-server-weather']}
[INFO] MCP server "filesystem": connected
[INFO] MCP server "filesystem": 11 tool(s) available:
[INFO] - read_file
[INFO] - read_multiple_files
[INFO] - write_file
[INFO] - edit_file
[INFO] - create_directory
[INFO] - list_directory
[INFO] - directory_tree
[INFO] - move_file
[INFO] - search_files
[INFO] - get_file_info
[INFO] - list_allowed_directories
[INFO] MCP server "fetch": connected
[INFO] MCP server "fetch": 1 tool(s) available:
[INFO] - fetch
[INFO] MCP server "weather": connected
[INFO] MCP server "weather": 2 tool(s) available:
[INFO] - get-alerts
[INFO] - get-forecast
[INFO] MCP servers initialized: 14 tool(

In [5]:
import dotenv
import getpass
import os

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [6]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [7]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    llm,
    tools
)


In [ ]:
from langchain.schema import HumanMessage

query = "What is the current weather in New York City?"
messages = [HumanMessage(content=query)]

result = await agent.ainvoke({"messages": messages})
print(result)
# the last message should be an AIMessage
response = result["messages"][-1].content
print(response)

[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] MCP tool "fetch"/"fetch" received input: {'url': 'https://www.weather.com/weather/today/l/Quer%C3%A9taro+Mexico?canonicalCityId=13b88238f38144c8ae7f88ecf5f8a428c32c1397b94df3133220079e35659243', 'max_length': 5000}
[WARNING] MCP tool "fetch"/"fetch" caused error:  Tool execution failed: [TextContent(type='text', text='Failed to fetch https://www.weather.com/weather/today/l/Quer%C3%A9taro+Mexico?canonicalCityId=13b88238f38144c8ae7f88ecf5f8a428c32c1397b94df3133220079e35659243 - status code 404', annotations=None)]
[INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[INFO] MCP tool "fetch"/"fetch" received input: {'url': 'https://www.weather.com/es-MX/clima/hoy/l/Querétaro+Querétaro+Mexico?canonicalCityId=13b88238f38144c8ae7f88ecf5f8a428c32c1397b94df3133220079e35659243', 'max_length': 5000}
[WARNING] MCP tool "fetch"/"fetch" caused error:  Tool execution failed:

{'messages': [HumanMessage(content='What is the current weather in Queretaro, Mexico?', additional_kwargs={}, response_metadata={}, id='ddd945e4-bdb0-4e59-9865-68a58ee3c494'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Bnm4BO72vizcjHFBg5XZESkf', 'function': {'arguments': '{"url":"https://www.weather.com/weather/today/l/Quer%C3%A9taro+Mexico?canonicalCityId=13b88238f38144c8ae7f88ecf5f8a428c32c1397b94df3133220079e35659243","max_length":5000}', 'name': 'fetch'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 1052, 'total_tokens': 1131, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_86d0290411', 'id': 'chatcmpl-BJ1TBM7FpAwiu0n53edzUUfIngjvK', 'finish_reason': 'tool_calls', '

In [13]:
import json

# Pretty-print the JSON structure of the result variable
print(json.dumps(result, indent=4, default=str))

{
    "messages": [
        "content='What is the current weather in New York City?' additional_kwargs={} response_metadata={} id='c7e48076-2147-4cab-ba92-de9f7a6a079c'",
        "content='' additional_kwargs={'tool_calls': [{'id': 'call_CxKxIrnSEhlpjOQmMsB0qi1W', 'function': {'arguments': '{\"latitude\":40.7128,\"longitude\":-74.006}', 'name': 'get-forecast'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 1050, 'total_tokens': 1076, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_86d0290411', 'id': 'chatcmpl-BJ1Jhhp8akBq2HApxzvOQ11odFazw', 'finish_reason': 'tool_calls', 'logprobs': None} id='run-853d9a2c-ed32-4edc-8b89-7e6f2aa604bc-0' tool_calls=[{'name': 'get-forecast', 'args': {'latitude': 40.7

In [ ]:
response